In [2]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR
import sentencepiece as spm
import math
import tqdm

random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Paths
CORPUS_DIR = "corpus"
MODEL_DIR = "models"
SUBSET_SIZE = 200000
EF_SIZE = 13000  # Approximate Endfield sentence count
WIKI_SUBSET_SIZE = SUBSET_SIZE - EF_SIZE

# Output paths for subset
SUBSET_SRC = os.path.join(CORPUS_DIR, "subset_train.skz")
SUBSET_TGT = os.path.join(CORPUS_DIR, "subset_train.zh")
SUBSET_VAL_SRC = os.path.join(CORPUS_DIR, "subset_val.skz")
SUBSET_VAL_TGT = os.path.join(CORPUS_DIR, "subset_val.zh")

Using device: cpu


## Build subset with all Endfield + sampled Wikipedia

In [3]:
def load_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

# Load Endfield data (must be fully included)
ef_zh = load_lines(os.path.join(CORPUS_DIR, "endfield.zh"))
ef_skz = load_lines(os.path.join(CORPUS_DIR, "endfield.skz"))
assert len(ef_zh) == len(ef_skz), "Endfield source/target mismatch"
print(f"Loaded Endfield: {len(ef_zh)} pairs")

# Load Wikipedia training data
wiki_zh = load_lines(os.path.join(CORPUS_DIR, "train.zh"))
wiki_skz = load_lines(os.path.join(CORPUS_DIR, "train.skz"))
assert len(wiki_zh) == len(wiki_skz), "Wikipedia source/target mismatch"
print(f"Loaded Wikipedia: {len(wiki_zh)} pairs")

# Sample Wikipedia subset
wiki_indices = random.sample(range(len(wiki_zh)), WIKI_SUBSET_SIZE)
wiki_sub_zh = [wiki_zh[i] for i in wiki_indices]
wiki_sub_skz = [wiki_skz[i] for i in wiki_indices]

# Combine: Endfield first, then Wikipedia subset
subset_zh = ef_zh + wiki_sub_zh
subset_skz = ef_skz + wiki_sub_skz

# Shuffle combined subset
combined = list(zip(subset_skz, subset_zh))
random.shuffle(combined)
subset_skz, subset_zh = zip(*combined)

# Write subset files
with open(SUBSET_SRC, "w", encoding="utf-8") as f_src, \
     open(SUBSET_TGT, "w", encoding="utf-8") as f_tgt:
    for skz, zh in zip(subset_skz, subset_zh):
        f_src.write(skz + "\n")
        f_tgt.write(zh + "\n")

# Reserve 5% of subset for validation
val_size = int(len(subset_zh) * 0.05)
val_skz, val_zh = subset_skz[:val_size], subset_zh[:val_size]
train_skz, train_zh = subset_skz[val_size:], subset_zh[val_size:]

with open(SUBSET_VAL_SRC, "w", encoding="utf-8") as f_src, \
     open(SUBSET_VAL_TGT, "w", encoding="utf-8") as f_tgt:
    for skz, zh in zip(val_skz, val_zh):
        f_src.write(skz + "\n")
        f_tgt.write(zh + "\n")

with open(SUBSET_SRC, "w", encoding="utf-8") as f_src, \
     open(SUBSET_TGT, "w", encoding="utf-8") as f_tgt:
    for skz, zh in zip(train_skz, train_zh):
        f_src.write(skz + "\n")
        f_tgt.write(zh + "\n")

print(f"Subset created: {len(train_zh)} train, {len(val_zh)} val pairs")
print(f"Endfield coverage: {len(ef_zh)}/{SUBSET_SIZE} ({100*len(ef_zh)/SUBSET_SIZE:.1f}%)")

Loaded Endfield: 13096 pairs
Loaded Wikipedia: 16999209 pairs
Subset created: 190092 train, 10004 val pairs
Endfield coverage: 13096/200000 (6.5%)


## Lightweight model config for rapid validation

In [4]:
# Hyperparameters (scaled down for fast iteration)
PAD_ID, SOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3
MAX_SEQ_LEN = 64
BATCH_SIZE = 64
EPOCHS = 200
LR = 5e-4
WARMUP_STEPS = 200
D_MODEL = 256
NHEAD = 4
NUM_LAYERS = 3
DIM_FEEDFORWARD = 512
DROPOUT = 0.1

# Load tokenizers (use existing merged models)
src_sp = spm.SentencePieceProcessor()
src_sp.Load(os.path.join(MODEL_DIR, "sp_merged_skz.model"))
tgt_sp = spm.SentencePieceProcessor()
tgt_sp.Load(os.path.join(MODEL_DIR, "sp_merged_zh.model"))

True

## Dataset and DataLoader

In [5]:
class SubsetDataset(Dataset):
    def __init__(self, src_path, tgt_path, max_len=MAX_SEQ_LEN):
        with open(src_path, encoding="utf-8") as f:
            self.src = f.readlines()
        with open(tgt_path, encoding="utf-8") as f:
            self.tgt = f.readlines()
        assert len(self.src) == len(self.tgt), "Source/target line count mismatch"
        self.max_len = max_len

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        s = [SOS_ID] + src_sp.encode(self.src[idx].strip())[:self.max_len-2] + [EOS_ID]
        t = [SOS_ID] + tgt_sp.encode(self.tgt[idx].strip())[:self.max_len-2] + [EOS_ID]
        return torch.tensor(s), torch.tensor(t)

def collate_fn(batch):
    src, tgt = zip(*batch)
    src_pad = nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=PAD_ID)
    tgt_pad = nn.utils.rnn.pad_sequence(tgt, batch_first=True, padding_value=PAD_ID)
    return src_pad, tgt_pad

train_ds = SubsetDataset(SUBSET_SRC, SUBSET_TGT)
val_ds = SubsetDataset(SUBSET_VAL_SRC, SUBSET_VAL_TGT)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_fn, num_workers=2)
print(f"DataLoader ready: {len(train_ds)} train, {len(val_ds)} val")

DataLoader ready: 190092 train, 10004 val


## Model Definition

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class SmallTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab, D_MODEL)
        self.tgt_emb = nn.Embedding(tgt_vocab, D_MODEL)
        self.pos_enc = PositionalEncoding(D_MODEL, DROPOUT)
        self.transformer = nn.Transformer(
            d_model=D_MODEL, nhead=NHEAD, num_encoder_layers=NUM_LAYERS,
            num_decoder_layers=NUM_LAYERS, dim_feedforward=DIM_FEEDFORWARD,
            dropout=DROPOUT, batch_first=True
        )
        self.out = nn.Linear(D_MODEL, tgt_vocab)
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        src = self.pos_enc(self.src_emb(src) * math.sqrt(D_MODEL))
        tgt = self.pos_enc(self.tgt_emb(tgt) * math.sqrt(D_MODEL))
        mem = self.transformer.encoder(src, src_key_padding_mask=(src==PAD_ID))
        out = self.transformer.decoder(tgt, mem, tgt_mask=tgt_mask,
                                       tgt_key_padding_mask=(tgt==PAD_ID),
                                       memory_key_padding_mask=(src==PAD_ID))
        return self.out(out)

model = SmallTransformer(src_sp.vocab_size(), tgt_sp.vocab_size()).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 12,195,456


## Training loop with validation

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.98), eps=1e-9)

def lr_lambda(step):
    step += 1
    return min(step * WARMUP_STEPS**-1.5, step**-0.5)
scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

def generate_bool_causal_mask(sz, device):
    return ~torch.tril(torch.ones(sz, sz, device=device, dtype=torch.bool))

def evaluate(loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            tgt_in = tgt[:, :-1]
            sz = tgt_in.size(1)
            tgt_mask = generate_bool_causal_mask(sz, device)
            out = model(src, tgt_in, src_mask=(src==PAD_ID), tgt_mask=tgt_mask)
            loss = criterion(out.transpose(1,2), tgt[:, 1:])
            total_loss += loss.item()
    model.train()
    return total_loss / len(loader)

print("Starting subset training...")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for src, tgt in tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        src, tgt = src.to(device), tgt.to(device)
        tgt_in = tgt[:, :-1]
        sz = tgt_in.size(1)
        tgt_mask = generate_bool_causal_mask(sz, device)
        
        optimizer.zero_grad()
        out = model(src, tgt_in, src_mask=(src==PAD_ID), tgt_mask=tgt_mask)
        loss = criterion(out.transpose(1,2), tgt[:, 1:])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
    
    val_loss = evaluate(val_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f}")

# Save subset model for inspection
torch.save(model.state_dict(), os.path.join(MODEL_DIR, "model_subset.pt"))
print("Subset training complete. Model saved.")

Starting subset training...


Epoch 1:   0%|          | 0/2971 [00:00<?, ?it/s]

## Quick inference test on Endfield samples

In [1]:
def greedy_decode(model, src_text, max_len=32):
    model.eval()
    src_ids = torch.tensor([SOS_ID] + src_sp.encode(src_text) + [EOS_ID], dtype=torch.long).unsqueeze(0).to(device)
    tgt_ids = torch.tensor([SOS_ID], dtype=torch.long).unsqueeze(0).to(device)
    
    for _ in range(max_len - 1):
        sz = tgt_ids.size(1)
        tgt_mask = generate_bool_causal_mask(sz, device)
        out = model(src_ids, tgt_ids, src_mask=(src_ids==PAD_ID), tgt_mask=tgt_mask)
        next_token = out[:, -1, :].argmax(dim=-1)
        tgt_ids = torch.cat([tgt_ids, next_token.unsqueeze(1)], dim=1)
        if next_token.item() == EOS_ID:
            break
    return tgt_sp.decode(tgt_ids.squeeze().tolist()[1:-1])

# Test on a few Endfield cipher samples
test_samples = ef_skz[:5]
print("Inference test on Endfield samples:")
for i, cipher in enumerate(test_samples):
    output = greedy_decode(model, cipher, max_len=40)
    print(f"{i+1}. Cipher: {cipher[:30]}...")
    print(f"   Output: {output}")
    print()

NameError: name 'ef_skz' is not defined